# Train nn.Transformer baseline + SentencePiece BPE on Tatoeba EN→DE

**Goal.** Train PyTorch's built-in `nn.Transformer` (Pre-LN) with the
same 8 k-vocab SentencePiece BPE tokenizers, under identical
hyperparameters as the hand-rolled run, so the comparison in §6.7 of
the paper is fair (same data, same optimizer, same schedule).

**Why two notebooks.** The hand-rolled and the baseline checkpoint
have different `model_type` tags (`hand_rolled` vs `baseline_nn`) and
different state_dict layouts. Keeping them in separate notebooks
avoids accidental overwrites and lets `translate.py` / `evaluate.py`
auto-dispatch via the `--baseline-nn` flag.

**Outputs persisted to Drive** (`MyDrive/transformer_runs/baseline_nn/`):
- `checkpoints/baseline_{best,latest}.pt`
- `checkpoints/baseline_metrics.jsonl`
- `.data/spm/spm_{en,de}.{model,vocab}` (reused from the BPE run)

## 1. Mount Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
REPO_DIR = '/content/transformer'
if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull --ff-only
else:
    !git clone https://github.com/oscar2697/transformer.git {REPO_DIR}
    %cd {REPO_DIR}

!pip install -q -r requirements.txt
!nvidia-smi -L

## 2. Link Drive and reuse SPM models

If the BPE run already finished, its SPM models live in Drive at
`MyDrive/transformer_runs/bpe/spm/`. We copy them in rather than
re-training.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/transformer_runs/baseline_nn'
BPE_ROOT   = '/content/drive/MyDrive/transformer_runs/bpe'
!mkdir -p {DRIVE_ROOT}

!rm -rf checkpoints
!ln -s {DRIVE_ROOT}/checkpoints checkpoints
!mkdir -p .data/spm

# Reuse SPM models from the BPE run so vocabularies match exactly.
if os.path.exists(f'{BPE_ROOT}/spm/spm_en.model'):
    !cp {BPE_ROOT}/spm/* .data/spm/
elif os.path.exists('.data/spm/spm_en.model'):
    print('SPM models already in repo.')
else:
    raise RuntimeError(
        'No SPM models found. Run scripts/colab/train_bpe_colab.ipynb first '
        '(or train SPM manually with config.SPM_VOCAB_SIZE = 8000).'
    )

!ls -la .data/spm/ checkpoints/

## 3. Training loop — chunked, with auto-resume

Same chunking strategy as the BPE notebook (5 epochs per Colab
session, `--resume checkpoints/baseline_latest.pt` on subsequent
sessions). The wrapper script `train_baseline_nn_colab.py` injects
`--baseline-nn` for us.

In [ ]:
# --- Configurable per session -------------------------------------------------
TOTAL_EPOCHS = 25
CHUNK        = 5
BATCH_SIZE   = 32
WARMUP       = 2000
# -----------------------------------------------------------------------------

import os, time
CKPT = 'checkpoints/baseline_latest.pt'
start_epoch = 0
if os.path.exists(CKPT):
    info = torch.load(CKPT, map_location='cpu', weights_only=False)
    start_epoch = info.get('epoch', -1) + 1
    print(f'Found baseline checkpoint at epoch {start_epoch}; resuming.')

target = min(start_epoch + CHUNK, TOTAL_EPOCHS)
print(f'Training epochs {start_epoch + 1}..{target}  (of {TOTAL_EPOCHS})')

if start_epoch >= TOTAL_EPOCHS:
    print('Already at target epoch count — nothing to do. Skip to §4.')
else:
    t0 = time.time()
    !python scripts/train_baseline_nn_colab.py \
        --epochs {target} \
        --batch-size {BATCH_SIZE} \
        --warmup {WARMUP} \
        --resume {CKPT if start_epoch > 0 else ''} \
        --log-interval 100 --val-interval 0
    print(f'Chunk finished in {(time.time() - t0) / 3600:.2f} h.')

## 4. Inspect progress

In [ ]:
import json
epochs_log = []
with open('checkpoints/baseline_metrics.jsonl') as f:
    for line in f:
        rec = json.loads(line)
        if rec.get('split') == 'epoch':
            epochs_log.append(rec)

print(f'{"epoch":>5} {"train":>10} {"val":>10} {"val_ppl":>10} {"time_h":>8}')
for r in epochs_log:
    print(f"{r['epoch']:>5} {r['train_loss']:>10.4f} {r['val_loss']:>10.4f} "
          f"{r['val_ppl']:>10.2f} {r['epoch_time']/3600:>8.2f}")

## 5. Plot the curves

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if epochs_log:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    xs = [r['epoch'] for r in epochs_log]
    ax1.plot(xs, [r['train_loss'] for r in epochs_log], label='train')
    ax1.plot(xs, [r['val_loss'] for r in epochs_log], label='val')
    ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend(); ax1.set_title('Baseline NN — loss')
    ax2.plot(xs, [r['val_ppl'] for r in epochs_log], color='C2')
    ax2.set_xlabel('epoch'); ax2.set_ylabel('val_ppl'); ax2.set_title('Baseline NN — val perplexity')
    fig.tight_layout()
    out = '/content/drive/MyDrive/transformer_runs/baseline_nn/training_curves_baseline.png'
    fig.savefig(out, dpi=120)
    print(f'Saved {out}')

## 6. Final step — package artifacts

Run only when `start_epoch >= TOTAL_EPOCHS`.

In [ ]:
import os
CKPT_META = 'checkpoints/baseline_latest.pt'
done = False
if os.path.exists(CKPT_META):
    info = torch.load(CKPT_META, map_location='cpu', weights_only=False)
    done = info.get('epoch', -1) + 1 >= TOTAL_EPOCHS

if not done:
    print('Training not finished yet — re-run §3 in a fresh session.')
else:
    %cd {DRIVE_ROOT}
    !tar czf artifacts_baseline.tar.gz checkpoints .data/spm training_curves_baseline.png 2>/dev/null || true
    !ls -lh artifacts_baseline.tar.gz
    print('Download artifacts_baseline.tar.gz from Drive.')

## After downloading

On your local machine:

```bash
cd ~/Documents/Workspace/IA/transformer
tar xzf /path/to/artifacts_baseline.tar.gz
# checkpoints/baseline_{best,latest}.pt + baseline_metrics.jsonl now live in checkpoints/.
python evaluate.py --checkpoint checkpoints/baseline_best.pt --baseline-nn --split test
python translate.py --checkpoint checkpoints/baseline_best.pt --baseline-nn --text "How are you?"
```

Fill §6.7 / Tabla 5 of the paper with the BLEU / chrF2 numbers, then
regenerate attention figures from the baseline checkpoint.